# Full-bill LLooM concept pipeline (8,317 bills)

Scales LLooM concept discovery to the full vote-linked corpus, checkpointing every
stage to **parquet** so you can stop, resume, and change clustering parameters
without repaying for Gemini embeddings. Same spirit as `lloom_experiment_v2.ipynb`.

The **interesting pipeline flow is here, in the cells**: the LLooM `distill_filter` /
`distill_summarize` / `synthesize` / `review` calls, the Gemini embedding loop, and
the seeded UMAP/HDBSCAN clustering are all explicit. `lawsofisrael.concepts` (`cp`)
only holds mechanical plumbing: parquet checkpoints, the document-selection policy,
the extraction/chunking wrappers, and the final bill-level aggregation.

Rules: **no KNN**, **no scoring**. Clusters are the result; concept labels are
semantic descriptions of clusters. Every paid call is content-keyed, so a rerun
skips rows already done.

## 0. Setup, paths, parameters

In [22]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import asyncio
import re
import pandas as pd

ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))

from config import (
    MODEL_CONFIG, MAX_CONTEXT_TOKENS, MAX_OUTPUT_TOKENS,
    GEMINI_EMBEDDING_MODEL, GEMINI_EMBEDDING_DIM,
    GEMINI_EMBEDDING_BASE_URL, GEMINI_EMBEDDING_API_KEY,
)
from lawsofisrael import concepts as cp, chunking, prompts_v2
from lawsofisrael import diagnostics_v2 as diag
from lawsofisrael.lloom import build_chat_model, build_gemini_embed_model, install_fence_tolerant_json_parser

# text_lloom primitives — called directly below so the flow is visible.
from text_lloom.concept_induction import synthesize, review
from text_lloom.llm import multi_query_gpt_wrapper
install_fence_tolerant_json_parser()

# --- Inputs (from the vote-scrape pipeline) --------------------------------
PREPARED = ROOT / "dataset" / "knesset_votes" / "prepared"
BILLS_PATH = PREPARED / "dim_bills.parquet"
DOCUMENTS_PATH = PREPARED / "dim_bill_documents.parquet"

# --- Run outputs (parquet checkpoints) -------------------------------------
RUN = ROOT / "dataset" / "bill_concepts" / "runs" / "full-v2"
DOCUMENT_STORE = ROOT / "dataset" / "bill_concepts" / "raw" / "documents" / "sha256"
RUN.mkdir(parents=True, exist_ok=True)
def path(name):
    return RUN / f"{name}.parquet"

# --- Parameters ------------------------------------------------------------
N_BILLS = 1000            # None = all 8,317; int = sample for iteration
# Whole-bill chunks (same budget as lloom_experiment_v2): the entire bill
# fits in one chunk so distill sees full-bill context, no per-section split.
CHUNK_TOKENS = chunking.compute_chunk_budget(MAX_CONTEXT_TOKENS, MAX_OUTPUT_TOKENS)
FILTER_N_QUOTES = 5
SUMMARY_N_BULLETS = 7
SYNTH_N_CONCEPTS = 1
SEED = 42
LLM_BATCH = 25
# LLooM already runs prompts inside each batch concurrently. Keep this at 2
# with a 100-RPM quota: two 25-chunk batches can issue 50 requests per phase.
DISTILL_BATCH_CONCURRENCY = 2
EMBED_DIM = GEMINI_EMBEDDING_DIM or 3072
EMBED_BATCH = 64
# API calls: raise only after confirming the provider quota has headroom.
EMBED_BATCH_CONCURRENCY = 2
SYNTHESIS_CLUSTER_CONCURRENCY = 2
# UMAP/HDBSCAN candidates are CPU/memory-heavy.
CLUSTER_CANDIDATE_CONCURRENCY = 2
# Knesset file server: retry entire streamed transfers, not only HTTP setup.
DOWNLOAD_MAX_ATTEMPTS = 5
DOWNLOAD_CONNECT_TIMEOUT_SECONDS = 10
DOWNLOAD_READ_TIMEOUT_SECONDS = 20
DOWNLOAD_RETRY_BACKOFF_SECONDS = 1.0
EMBED_INSTRUCTION = diag.DEFAULT_EMBED_INSTRUCTION
USE_DOMAIN_CONTEXT = True

# Seeded UMAP/HDBSCAN candidates (local; cheap to re-run from saved vectors).
CLUSTER_CANDIDATES = {
    "A": dict(hdbscan_min_cluster_size=3, hdbscan_cluster_selection_method="leaf"),
    "B": dict(hdbscan_min_cluster_size=5, hdbscan_cluster_selection_method="eom"),
    "C": dict(hdbscan_min_cluster_size=5, hdbscan_cluster_selection_method="leaf"),
    "D": dict(hdbscan_min_cluster_size=7, hdbscan_cluster_selection_method="eom"),
    "E": dict(hdbscan_min_cluster_size=4, hdbscan_cluster_selection_method="leaf"),
}
# Inspect one candidate below, then set CHOSEN_CANDIDATE to synthesize labels.
INSPECT_CANDIDATE = "E"
CHOSEN_CANDIDATE = "E"

DISTILL_PROTOCOL_VERSION = "tagged-v2-most-important"
FILTER_DELIMITED_PROMPT = """
I have the following TEXT EXAMPLE of Israeli legislative text:
{ex}

Extract up to {n_quotes} exact quotes that capture the most important operative
legal content in this example.
Copy quote text exactly. For each quote, output only:
<QUOTE>
the exact quote
</QUOTE>
If there are no relevant quotes, output exactly <NO_QUOTES/>.
Do not use JSON, Markdown code fences, numbering, or any other text.
"""
SUMMARIZE_DELIMITED_PROMPT = prompts_v2.DISTILL_SUMMARIZE_PROMPT.split(
    "Please respond ONLY with a valid JSON", 1)[0] + """
Return each bullet only between these exact delimiters:
<BULLET>
the bullet
</BULLET>
If no operative legal rule is present, output exactly <NO_BULLETS/>.
Do not use JSON, Markdown code fences, numbering, or any other text.
"""
SUMMARIZE_INFO = prompts_v2.distill_summarize_prompt_info(SUMMARIZE_DELIMITED_PROMPT)
SUMMARIZE_INFO["summarize_prompt_version"] += "-delimited-most-important"
SYNTH_INFO = prompts_v2.synthesize_prompt_info()
print("Run dir:", RUN)
print("Summarize prompt:", SUMMARIZE_INFO, "| Synthesize prompt:", SYNTH_INFO)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Run dir: /home/nick/Documents/Projects/lawsofisrael/dataset/bill_concepts/runs/full-v2
Summarize prompt: {'summarize_prompt_version': 'v2.1.2-delimited-most-important', 'summarize_prompt_hash': '55d1ad51508c'} | Synthesize prompt: {'synthesize_prompt_version': 'v2.0.0', 'synthesize_prompt_hash': 'c141b3edbbe5'}


## 1. Load inputs and select one document per bill

Selection policy (official enacted law first, otherwise the latest available bill
draft from the allowlisted `LegalDocuments` / `DraftLaws` collections) lives in
`cp.select_documents`. Background and procedural document collections are excluded.

In [12]:
bills = pd.read_parquet(BILLS_PATH)
if N_BILLS:
    bills = bills.head(N_BILLS).copy()
documents = pd.read_parquet(DOCUMENTS_PATH)
if not {"LegalDocuments", "DraftLaws"}.issubset(set(documents.get("source_field", []))):
    raise RuntimeError(
        "dim_bill_documents.parquet is the legacy LegalDocuments-only table. "
        "Rerun scrape_passed_bills.ipynb build/report cells with CACHE_ONLY=True first.")

selection = cp.select_documents(bills, documents)
cp.write_parquet(selection, path("document_selection"))
print(selection["selection_state"].value_counts().to_dict())
selection.head()

{'selected': 991, 'no_supported_document': 9}


,bill_id,bill_document_id,source_field,document_label,document_url,document_format,document_date,selected_stage,selection_state,selection_rule,selection_reason,candidate_count,selection_rank
0,15752,doc_6e4f4b2e631d780922db5665,LegalDocuments,חוק - פרסום ברשומות,https://fs.knesset.gov.il//16/law/16_lsr_29959...,pdf,None,official_law,selected,legal_stage_then_format,selected official_law (pdf) from 2 eligible ca...,2,0.0
1,16127,doc_1a545d364e83123413b4320d,LegalDocuments,חוק - פרסום ברשומות,https://fs.knesset.gov.il//17/law/17_lsr_29960...,pdf,None,official_law,selected,legal_stage_then_format,selected official_law (pdf) from 2 eligible ca...,2,0.0
2,16403,doc_b95374546ab77f08a124c5e3,LegalDocuments,חוק - פרסום ברשומות,https://fs.knesset.gov.il//16/law/16_lsr_29960...,pdf,None,official_law,selected,legal_stage_then_format,selected official_law (pdf) from 2 eligible ca...,2,0.0
3,16634,doc_4a360a4125c0717b512d328c,LegalDocuments,חוק - פרסום ברשומות,https://fs.knesset.gov.il//16/law/16_lsr_30099...,pdf,None,official_law,selected,legal_stage_then_format,selected official_law (pdf) from 2 eligible ca...,2,0.0
4,16639,doc_2a69a34cd0d6b7ce19c110c4,DraftLaws,הצעת חוק לדיון מוקדם,https://fs.knesset.gov.il//16/law/16_lst_59401...,doc,None,preliminary_reading,selected,legal_stage_then_format,selected preliminary_reading (doc) from 1 elig...,1,4.0


## 2. Download selected PDF/DOCX/DOC/RTF files (local, content-addressed) — network

In [13]:
document_manifest = cp.download_documents(
    selection, DOCUMENT_STORE, prior=cp.read_parquet(path("document_manifest")),
    max_attempts=DOWNLOAD_MAX_ATTEMPTS,
    connect_timeout_seconds=DOWNLOAD_CONNECT_TIMEOUT_SECONDS,
    read_timeout_seconds=DOWNLOAD_READ_TIMEOUT_SECONDS,
    retry_backoff_seconds=DOWNLOAD_RETRY_BACKOFF_SECONDS,
)
cp.write_parquet(document_manifest, path("document_manifest"))
print(document_manifest["download_state"].value_counts().to_dict())

{'available': 991, 'not_selected': 9}


## 3. Extract documents and chunk (local, provenance-preserving)

In [14]:
ext_docs, ext_blocks = cp.extract_documents(
    document_manifest, bills,
    prior_documents=cp.read_parquet(path("extraction_documents")),
    prior_blocks=cp.read_parquet(path("extraction_blocks")),
)
cp.write_parquet(ext_docs, path("extraction_documents"))
cp.write_parquet(ext_blocks, path("extraction_blocks"))

chunks = cp.build_chunks(ext_docs, ext_blocks, bills, chunk_tokens=CHUNK_TOKENS)
cp.write_parquet(chunks, path("chunks"))
print("extracted:", (ext_docs["extraction_state"] == "success").sum(),
      "blocks:", len(ext_blocks), "chunks:", len(chunks))
chunks[["chunk_id", "bill_id", "bill_title", "page_no", "tokens"]].head()

Extracting documents:   0%|          | 0/1000 [00:00<?, ?document/s]

Building chunks:   0%|          | 0/991 [00:00<?, ?document/s]

extracted: 991 blocks: 68569 chunks: 991


,chunk_id,bill_id,bill_title,page_no,tokens
0,15752:22db5665:0000,15752,"חוק רשות הספנות והנמלים, התשס""ד-2004",1.0,42585
1,16127:13b4320d:0000,16127,"חוק להגנת חיית הבר (תיקון מס' 6), התשס""ט-2008",1.0,1567
2,16403:a124c5e3:0000,16403,"חוק ההוצאה לפועל (תיקון מס' 24) (תיקון), התשס""...",1.0,509
3,16634:512d328c:0000,16634,"חוק גיל פרישה, התשס""ד-2004",1.0,29723
4,16639:19c110c4:0000,16639,"הצעת חוק נכי רדיפות הנאצים (תיקון - כפל קצבה),...",NaN,1236


## 4. Distill — LLooM quote filter + condition-faithful summaries (paid)

This is LLooM's `distill_filter` then `distill_summarize`, called directly. The only
thing wrapped around them is a content-key cache: a chunk already summarized in
`bullets.parquet` is skipped, and each batch is written before the next so an
interrupted run keeps its progress. We build the chat model exactly like v2.

In [15]:
chat_model = build_chat_model(
    model_config=MODEL_CONFIG, base_url=MODEL_CONFIG["base_url"],
    api_key=MODEL_CONFIG["api_key"], max_output_tokens=MAX_OUTPUT_TOKENS,
)

bullets = cp.read_parquet(path("bullets"))
distill_status = cp.read_parquet(path("distill_status"))
completed_states = {"success", "no_usable_quotes"}
done_chunk_keys = (set(distill_status.loc[
    distill_status["distill_state"].isin(completed_states)
    & distill_status.get("distill_protocol_version", pd.Series("", index=distill_status.index)).eq(DISTILL_PROTOCOL_VERSION),
    "chunk_key"].astype(str))
    if distill_status is not None and not distill_status.empty else set())
# Only reuse bullets made by the active delimiter protocol. Older JSON and
# RELATED TO 42 runs are deliberately invalidated and regenerated.
if bullets is not None and "distill_protocol_version" in bullets:
    bullets = bullets[bullets["distill_protocol_version"].eq(DISTILL_PROTOCOL_VERSION)].copy()
else:
    bullets = None
todo = chunks[~chunks["chunk_key"].astype(str).isin(done_chunk_keys)]
print(f"{len(todo)} chunks to distill ({len(done_chunk_keys)} completed checkpoints)")
chunk_key_by_id = chunks.set_index("chunk_id")["chunk_key"].to_dict()

bullet_rows = [] if bullets is None else bullets.to_dict("records")
status_rows = [] if distill_status is None else distill_status.to_dict("records")
batches = [todo.iloc[start:start + LLM_BATCH] for start in range(0, len(todo), LLM_BATCH)]
semaphore = asyncio.Semaphore(DISTILL_BATCH_CONCURRENCY)

def parse_delimited(response, tag, empty_marker):
    text = str(response or "").strip()
    values = [value.strip() for value in re.findall(
        rf"<{tag}>\s*(.*?)\s*</{tag}>", text, flags=re.DOTALL) if value.strip()]
    if values:
        return values, None
    if empty_marker in text:
        return [], "empty"
    return [], "malformed"

async def distill_batch(batch_index, batch):
    batch_chunks = batch[["chunk_id", "chunk_key"]].to_dict("records")
    async with semaphore:
        try:
            filter_args = [{
                "ex": row.text, "n_quotes": FILTER_N_QUOTES,
            } for row in batch[["chunk_id", "text"]].itertuples(index=False)]
            filter_responses, _ = await multi_query_gpt_wrapper(
                FILTER_DELIMITED_PROMPT, filter_args, chat_model)
            filtered_rows, statuses = [], {}
            for item, response in zip(batch_chunks, filter_responses):
                quotes, outcome = parse_delimited(response, "QUOTE", "<NO_QUOTES/>")
                if quotes:
                    filtered_rows.append({"chunk_id": item["chunk_id"], "text": "\n\n".join(quotes)})
                elif outcome == "empty":
                    statuses[item["chunk_id"]] = ("no_usable_quotes", None)
                else:
                    statuses[item["chunk_id"]] = ("failed", "filter response missing quote delimiters")
            if filtered_rows:
                filtered = pd.DataFrame(filtered_rows)
                summarize_args = [{
                    "ex": row.text, "seeding_phrase": "",
                    "n_bullets": SUMMARY_N_BULLETS, "n_words": "5-8",
                } for row in filtered.itertuples(index=False)]
                summary_responses, _ = await multi_query_gpt_wrapper(
                    SUMMARIZE_DELIMITED_PROMPT, summarize_args, chat_model)
                rows = []
                for row, response in zip(filtered.itertuples(index=False), summary_responses):
                    bullets_for_chunk, outcome = parse_delimited(response, "BULLET", "<NO_BULLETS/>")
                    if outcome == "malformed":
                        statuses[row.chunk_id] = ("failed", "summarize response missing bullet delimiters")
                        continue
                    statuses[row.chunk_id] = ("success", None)
                    for ordinal, text in enumerate(bullets_for_chunk):
                        rows.append({
                            "bullet_row_id": f"{row.chunk_id}#b{ordinal:03d}", "chunk_id": row.chunk_id,
                            "chunk_key": chunk_key_by_id[row.chunk_id], "bullet_ordinal": ordinal, "bullet": text,
                            "distill_protocol_version": DISTILL_PROTOCOL_VERSION,
                            "summarize_prompt_version": SUMMARIZE_INFO["summarize_prompt_version"],
                            "summarize_prompt_hash": SUMMARIZE_INFO["summarize_prompt_hash"],
                        })
            else:
                rows = []
            status_rows_for_batch = [{
                "chunk_id": item["chunk_id"], "chunk_key": item["chunk_key"],
                "distill_state": statuses.get(item["chunk_id"], ("failed", "missing batch outcome"))[0],
                "error": statuses.get(item["chunk_id"], ("failed", "missing batch outcome"))[1],
            } for item in batch_chunks]
            return batch_index, status_rows_for_batch, rows
        except Exception as error:  # noqa: BLE001 - checkpoint every chunk for retry, continue others
            return batch_index, [{
                "chunk_id": item["chunk_id"], "chunk_key": item["chunk_key"],
                "distill_state": "failed", "error": f"{type(error).__name__}: {error}",
            } for item in batch_chunks], []

from tqdm.auto import tqdm
tasks = [asyncio.create_task(distill_batch(index, batch)) for index, batch in enumerate(batches)]
completed_batches = completed_chunks = failed_batches = 0
progress = tqdm(total=len(batches), desc="Distilling chunks", unit="batch")
for task in asyncio.as_completed(tasks):
    _, batch_statuses, rows = await task
    status_rows.extend({**item, "distill_protocol_version": DISTILL_PROTOCOL_VERSION,
                        "completed_at": cp.now_iso()} for item in batch_statuses)
    distill_status = (pd.DataFrame(status_rows).drop_duplicates("chunk_key", keep="last")
                      .sort_values(["chunk_id", "chunk_key"], kind="stable")
                      .reset_index(drop=True))
    cp.write_parquet(distill_status, path("distill_status"))
    if rows:
        bullet_rows.extend(rows)
        bullets = (pd.DataFrame(bullet_rows).drop_duplicates("bullet_row_id", keep="last")
                   .sort_values(["chunk_id", "bullet_ordinal", "bullet_row_id"], kind="stable")
                   .reset_index(drop=True))
        cp.write_parquet(bullets, path("bullets"))
    completed_batches += 1
    completed_chunks += len(batch_statuses)
    failed_batches += any(item["distill_state"] == "failed" for item in batch_statuses)
    progress.update(1)
    progress.set_postfix(chunks=f"{completed_chunks}/{len(todo)}", bullets=len(bullet_rows), failed=failed_batches)
progress.close()

if bullets is None:
    bullets = pd.DataFrame(columns=["bullet_row_id", "chunk_id", "chunk_key", "bullet_ordinal", "bullet"])
print("total bullets:", len(bullets),
      "| chunks with >=1 bullet:", bullets["chunk_id"].nunique(), "of", len(chunks))

991 chunks to distill (0 completed checkpoints)


Distilling chunks:   0%|          | 0/40 [00:00<?, ?batch/s]

total bullets: 5728 | chunks with >=1 bullet: 957 of 991


## 5. Embed bullets with Gemini (paid)

Compose the exact embedding input (instruction + bill title/heading + bullet) with
`diag.build_embedding_input`, key each vector by input+model+dim, and only call
Gemini for bullets not already embedded. Re-clustering never re-embeds.

In [17]:
embed_model = build_gemini_embed_model(
    base_url=GEMINI_EMBEDDING_BASE_URL, api_key=GEMINI_EMBEDDING_API_KEY,
    model_name=GEMINI_EMBEDDING_MODEL, dimensions=EMBED_DIM, batch_size=EMBED_BATCH,
)
context_by_chunk = chunks.set_index("chunk_id")[["bill_title", "heading_path"]].to_dict("index")

def embed_input(row):
    ctx = context_by_chunk.get(row["chunk_id"]) if USE_DOMAIN_CONTEXT else None
    # make_chunk_lookup-style context: build_embedding_input reads name/heading_path
    ctx = {"name": ctx["bill_title"], "heading_path": ctx["heading_path"]} if ctx else None
    return diag.build_embedding_input(row["bullet"], instruction=EMBED_INSTRUCTION, context=ctx)

emb_inputs = bullets.copy()
emb_inputs["embedding_input"] = emb_inputs.apply(embed_input, axis=1)
emb_inputs["embedding_request_key"] = emb_inputs["embedding_input"].map(
    lambda t: cp.content_hash({"text": t, "model": GEMINI_EMBEDDING_MODEL, "dim": EMBED_DIM}))
cp.write_parquet(emb_inputs[["embedding_request_key", "bullet_row_id", "chunk_id", "embedding_input"]],
                 path("embedding_inputs"))

embeddings = cp.read_parquet(path("embeddings"))
done_keys = set(embeddings["embedding_request_key"]) if embeddings is not None else set()
todo = emb_inputs[~emb_inputs["embedding_request_key"].isin(done_keys)]
print(f"{len(todo)} bullets to embed ({len(done_keys)} cached)")

from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

rows = [] if embeddings is None else embeddings.to_dict("records")
batches = [todo.iloc[start:start + EMBED_BATCH] for start in range(0, len(todo), EMBED_BATCH)]
completed_batches = completed_bullets = 0
progress = tqdm(total=len(batches), desc="Embedding bullets", unit="batch")
with ThreadPoolExecutor(max_workers=EMBED_BATCH_CONCURRENCY, thread_name_prefix="embed") as executor:
    pending = {
        executor.submit(embed_model.fn, embed_model, batch["embedding_input"].tolist()): batch
        for batch in batches
    }
    for future in as_completed(pending):
        batch = pending[future]
        vectors, _ = future.result()  # Gemini call
        if len(vectors) != len(batch):
            raise RuntimeError(f"embedding response length {len(vectors)} != batch length {len(batch)}")
        for (_, item), vector in zip(batch.iterrows(), vectors):
            rows.append({"embedding_request_key": item["embedding_request_key"],
                         "bullet_row_id": item["bullet_row_id"], "vector": list(map(float, vector))})
        embeddings = (pd.DataFrame(rows).drop_duplicates("embedding_request_key", keep="last")
                      .sort_values(["embedding_request_key", "bullet_row_id"], kind="stable")
                      .reset_index(drop=True))
        cp.write_parquet(embeddings, path("embeddings"))
        completed_batches += 1
        completed_bullets += len(batch)
        progress.update(1)
        progress.set_postfix(bullets=f"{completed_bullets}/{len(todo)}", cached=len(embeddings))
progress.close()
print("total embeddings:", len(embeddings))

2784 bullets to embed (2944 cached)


Embedding bullets:   0%|          | 0/44 [00:00<?, ?batch/s]

total embeddings: 5727


## 6. Cluster — seeded UMAP + HDBSCAN (local)

`diag.run_clustering` (seeded, reproducible) over the saved Gemini vectors, once per
candidate. Inspect the summary (noise, cross-bill vs single-bill), then set
`CHOSEN_CANDIDATE` in cell 0. No API calls here, so re-run freely.

In [18]:
import numpy as np

aligned = (emb_inputs[["bullet_row_id", "embedding_request_key", "chunk_id"]]
           .merge(embeddings, on=["embedding_request_key", "bullet_row_id"], how="inner")
           .sort_values("bullet_row_id", kind="stable").reset_index(drop=True))
ids = aligned["bullet_row_id"].tolist()
vectors = np.vstack(aligned["vector"].map(lambda v: np.asarray(v, dtype=np.float32)))
bill_of = {bid: cid.split(":", 1)[0] for bid, cid in zip(aligned["bullet_row_id"], aligned["chunk_id"].astype(str))}
data_hash = cp.content_hash(aligned["embedding_request_key"].tolist())[:12]

from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

def run_candidate(key, params):
    cfg = diag.ClusterConfig(umap_n_neighbors=15, umap_n_components=5, umap_min_dist=0.0,
                             random_state=SEED, **params)
    res = diag.run_clustering(ids, [""] * len(ids), vectors, cfg)
    return key, f"{key}_{cfg.run_id()}_{data_hash}", res

results = {}
progress = tqdm(total=len(CLUSTER_CANDIDATES), desc="Clustering candidates", unit="candidate")
with ThreadPoolExecutor(max_workers=CLUSTER_CANDIDATE_CONCURRENCY, thread_name_prefix="cluster") as executor:
    pending = {executor.submit(run_candidate, key, params): key for key, params in CLUSTER_CANDIDATES.items()}
    for future in as_completed(pending):
        key, run_id, res = future.result()
        results[key] = (run_id, res)
        progress.update(1)
        progress.set_postfix(candidate=key, clusters=res.n_clusters(), noise=f"{res.noise_ratio():.0%}")
progress.close()

assign_rows, summary_rows = [], []
for key in CLUSTER_CANDIDATES:
    run_id, res = results[key]
    for i, bid in enumerate(ids):
        assign_rows.append({"candidate_key": key, "cluster_run_id": run_id, "bullet_row_id": bid,
                            "cluster_id": int(res.labels[i]), "is_noise": int(res.labels[i]) == -1,
                            "membership_probability": float(res.probabilities[i]),
                            "outlier_score": float(res.outlier_scores[i])})
    for comp in res.cluster_composition():
        summary_rows.append({"candidate_key": key, "cluster_run_id": run_id, **comp})
    print(f"[{key}] {run_id}: clusters={res.n_clusters()} noise={res.noise_ratio():.0%} "
          f"cross_bill={res.n_cross_bill_clusters()} single_bill={res.n_single_bill_clusters()}")

assignments = pd.DataFrame(assign_rows)
cp.write_parquet(assignments, path("cluster_assignments"))
cp.write_parquet(pd.DataFrame(summary_rows), path("cluster_summaries"))

Clustering candidates:   0%|          | 0/5 [00:00<?, ?candidate/s]

/home/nick/Documents/Projects/lawsofisrael/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/nick/Documents/Projects/lawsofisrael/.venv/lib/python3.12/site-packages/numba/np/ufunc/dufunc.py:343: NumbaWarning: Compilation requested for previously compiled argument types ((float32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
  warnings.warn(msg, errors.NumbaWarning)
/home/nick/Documents/Projects/lawsofisrael/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/nick/Documents/Projects/lawsofisrael/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/nick/Documents/Proj

[A] A_umap15x5_mcs3_leaf_seed42_8425a39a5859: clusters=636 noise=16% cross_bill=402 single_bill=234
[B] B_umap15x5_mcs5_eom_seed42_8425a39a5859: clusters=417 noise=11% cross_bill=317 single_bill=100
[C] C_umap15x5_mcs5_leaf_seed42_8425a39a5859: clusters=443 noise=14% cross_bill=337 single_bill=106
[D] D_umap15x5_mcs7_eom_seed42_8425a39a5859: clusters=267 noise=18% cross_bill=267 single_bill=0
[E] E_umap15x5_mcs4_leaf_seed42_8425a39a5859: clusters=521 noise=15% cross_bill=356 single_bill=165


PosixPath('/home/nick/Documents/Projects/lawsofisrael/dataset/bill_concepts/runs/full-v2/cluster_summaries.parquet')

## 7. Explore clustering candidates and choose deliberately

This is a local, no-API decision point. Compare candidate noise and bill coverage,
then inspect representative bullets from distinct bills in `INSPECT_CANDIDATE`.
Set `CHOSEN_CANDIDATE` in setup only after reviewing this evidence. Synthesis is
disabled while it remains `None`.

In [23]:
cluster_summaries = cp.read_parquet(path("cluster_summaries"))
if cluster_summaries is None or cluster_summaries.empty:
    raise RuntimeError("Run the clustering cell before exploration.")

candidate_comparison = (
    assignments.groupby(["candidate_key", "cluster_run_id"], as_index=False)
    .agg(
        n_bullets=("bullet_row_id", "size"),
        noise_ratio=("is_noise", "mean"),
        mean_membership_probability=("membership_probability", "mean"),
        mean_outlier_score=("outlier_score", "mean"),
    )
    .merge(
        cluster_summaries.groupby(["candidate_key", "cluster_run_id"], as_index=False)
        .agg(
            n_clusters=("cluster_id", "nunique"),
            cross_bill_clusters=("single_bill", lambda s: int(s.eq(False).sum())),
            single_bill_clusters=("single_bill", lambda s: int(s.eq(True).sum())),
            mean_cluster_bills=("n_bills", "mean"),
            mean_cluster_probability=("mean_prob", "mean"),
        ),
        on=["candidate_key", "cluster_run_id"], how="left",
    )
    .sort_values("candidate_key", kind="stable")
    .reset_index(drop=True)
)
display(candidate_comparison)

inspection = candidate_comparison.loc[
    candidate_comparison["candidate_key"].eq(INSPECT_CANDIDATE)
]
if len(inspection) != 1:
    raise ValueError(f"Unknown or ambiguous INSPECT_CANDIDATE: {INSPECT_CANDIDATE!r}")
inspection_run_id = inspection.iloc[0]["cluster_run_id"]
inspected_assignments = assignments.loc[
    assignments["cluster_run_id"].eq(inspection_run_id)
].copy()

# Read the strongest multi-bill clusters first; single-bill clusters remain
# visible in the comparison but are not the first evidence shown.
inspection_clusters = (
    cluster_summaries.loc[cluster_summaries["cluster_run_id"].eq(inspection_run_id)]
    .sort_values(["n_bills", "size", "mean_prob", "cluster_id"],
                 ascending=[False, False, False, True], kind="stable")
    .head(12)
)
display(inspection_clusters[["cluster_id", "size", "n_bills", "single_bill", "mean_prob"]])

evidence = (
    inspected_assignments.merge(
        inspection_clusters[["cluster_id", "size", "n_bills", "single_bill", "mean_prob"]],
        on="cluster_id", how="inner",
    )
    .merge(bullets[["bullet_row_id", "chunk_id", "bullet"]], on="bullet_row_id", how="left")
    .merge(chunks[["chunk_id", "bill_id", "bill_title", "heading_path", "page_no"]],
           on="chunk_id", how="left", validate="many_to_one")
    .sort_values(["n_bills", "size", "cluster_id", "membership_probability", "outlier_score", "bullet_row_id"],
                 ascending=[False, False, True, False, True, True], kind="stable")
)
representative_bullets = (
    evidence.drop_duplicates(["cluster_id", "bill_id"])
    .groupby("cluster_id", group_keys=False, sort=False).head(3)
)
display(representative_bullets[[
    "cluster_id", "size", "n_bills", "single_bill", "bill_id", "bill_title",
    "heading_path", "page_no", "bullet_row_id", "bullet",
    "membership_probability", "outlier_score",
]])

if CHOSEN_CANDIDATE is None:
    chosen = None
    chosen_run_id = None
    print("Exploratory state: set CHOSEN_CANDIDATE in setup after inspection; synthesis is disabled.")
else:
    selection = candidate_comparison.loc[candidate_comparison["candidate_key"].eq(CHOSEN_CANDIDATE)]
    if len(selection) != 1:
        raise ValueError(f"Unknown or ambiguous CHOSEN_CANDIDATE: {CHOSEN_CANDIDATE!r}")
    chosen_run_id = selection.iloc[0]["cluster_run_id"]
    chosen = assignments.loc[assignments["cluster_run_id"].eq(chosen_run_id)].copy()
    if not chosen["is_noise"].eq(False).any():
        raise ValueError(f"{CHOSEN_CANDIDATE!r} is all noise; choose another candidate.")
    print(f"Chosen {CHOSEN_CANDIDATE}: {chosen_run_id} | real clusters: "
          f"{chosen.loc[~chosen["is_noise"], "cluster_id"].nunique()}")

,candidate_key,cluster_run_id,n_bullets,noise_ratio,mean_membership_probability,mean_outlier_score,n_clusters,cross_bill_clusters,single_bill_clusters,mean_cluster_bills,mean_cluster_probability
0,A,A_umap15x5_mcs3_leaf_seed42_8425a39a5859,5727,0.159246,0.649445,0.317342,636,402,234,2.281447,0.828292
1,B,B_umap15x5_mcs5_eom_seed42_8425a39a5859,5727,0.114196,0.697045,0.304978,417,317,100,3.309353,0.833384
2,C,C_umap15x5_mcs5_leaf_seed42_8425a39a5859,5727,0.140911,0.674586,0.304978,443,337,106,3.097065,0.840660
3,D,D_umap15x5_mcs7_eom_seed42_8425a39a5859,5727,0.176881,0.626216,0.359926,267,267,0,4.808989,0.815402
4,E,E_umap15x5_mcs4_leaf_seed42_8425a39a5859,5727,0.150515,0.659848,0.313820,521,356,165,2.675624,0.831847


,cluster_id,size,n_bills,single_bill,mean_prob
1763,518,43,30,False,0.667301
1764,497,34,18,False,0.451725
1765,501,24,18,False,0.874986
1766,517,21,17,False,0.674478
1767,476,17,13,False,0.874574
1768,515,17,13,False,0.785102
1769,469,21,12,False,0.477982
1770,516,18,12,False,0.669347
1771,22,24,11,False,0.501461
1772,170,27,10,False,0.512087


,cluster_id,size,n_bills,single_bill,bill_id,bill_title,heading_path,page_no,bullet_row_id,bullet,membership_probability,outlier_score
59,518,43,30,False,131484,"הצעת חוק זכויות תלמידים מחוננים ומצטיינים, התש...",,NaN,131484:d2dfe025:0000#b005,אחריות השר על ביצוע הוראות חוק זה,1.000000,0.000000
78,518,43,30,False,132339,הצעת חוק מתן שירות בשפה הרוסית במשרדים ממשלתיי...,,NaN,132339:289941b6:0000#b002,הסמכת שר הפנים להיות ממונה על ביצוע החוק,1.000000,0.000000
226,518,43,30,False,80355,הצעת חוק למדיניות לאומית לשמירה על שטחים פתוחי...,,NaN,80355:d80bba2a:0000#b006,הסמכת שר הפנים לבצע את החוק ולהתקין תקנות בהתי...,1.000000,0.000000
30,497,34,18,False,130046,הצעת חוק רשות השידור (תיקון - ביטול אגרת רדיו ...,,NaN,130046:b600d701:0000#b000,ביטול סעיפים 28א עד 29ג בחוק רשות השידור,1.000000,0.000000
164,497,34,18,False,30584,הצעת חוק רשות השידור (תיקון - ביטול אגרת טלויז...,,NaN,30584:4850df9f:0000#b000,ביטול סעיף 28א לחוק רשות השידור,1.000000,0.000000
141,497,34,18,False,28094,הצעת חוק רשות השידור (תיקון - ביטול אגרת הטלוי...,,NaN,28094:feec4dbc:0000#b000,ביטול סעיפים 28א עד 29ג בחוק רשות השידור,0.938634,0.061366
58,501,24,18,False,131284,"חוק המרשם הפלילי ותקנת השבים (תיקון מס' 8), הת...",,1.0,131284:d0c03777:0000#b004,איסור השגה או דרישה של מידע מן המרשם במישרין א...,1.000000,0.000000
201,501,24,18,False,74965,"חוק הגבלת שימוש במקום לשם מניעת ביצוע עבירות, ...",,1.0,74965:52290d95:0000#b004,המשתמש או המרשה שימוש במקום בניגוד לצו דינו מא...,1.000000,0.000000
203,501,24,18,False,75262,חוק מניעת זיהום הים ממקורות יבשתיים (תיקון מס'...,,1.0,75262:6e16208c:0000#b001,העושה עבירה לפי חוק זה דינו מאסר שנה או קנס מוגדל,1.000000,0.000000
80,517,21,17,False,132394,"הצעת חוק להגבלת שכר הבכירים, התשס""ז-2006",,NaN,132394:2428f3d9:0000#b002,הסמכת שר האוצר על ביצוע חוק זה,1.000000,0.000000


Chosen E: E_umap15x5_mcs4_leaf_seed42_8425a39a5859 | real clusters: 521


## 8. Synthesize concept labels for the chosen clustering — LLooM synthesize + review (paid, no scoring)

In [26]:
if chosen is None:
    raise RuntimeError("No CHOSEN_CANDIDATE: inspect the exploration cell and select one before synthesis. No API calls were made.")
run_id = chosen_run_id
# Rebuild from artifacts so this cell is safe to re-run regardless of kernel state.
chosen = assignments.loc[assignments["cluster_run_id"].eq(run_id)].copy()
chosen = chosen.merge(bullets[["bullet_row_id", "bullet"]], on="bullet_row_id", how="left")
print("chosen:", run_id, "| clusters:", chosen.loc[chosen.cluster_id != -1, "cluster_id"].nunique())

concepts = cp.read_parquet(path("concepts"))
concept_rows = [] if concepts is None else concepts.to_dict("records")
done_clusters = set((concepts["cluster_run_id"] + ":" + concepts["cluster_id"].astype(str))) if concepts is not None else set()

from tqdm.auto import tqdm

pending_groups = [
    (int(cluster_id), group)
    for cluster_id, group in chosen[chosen.cluster_id != -1].groupby("cluster_id", sort=True)
    if f"{run_id}:{cluster_id}" not in done_clusters
]
semaphore = asyncio.Semaphore(SYNTHESIS_CLUSTER_CONCURRENCY)

async def synthesize_cluster(cluster_id, group):
    async with semaphore:
        # LLooM synthesize expects [doc_id, doc, cluster_id].
        cluster_df = group[["bullet_row_id", "bullet"]].rename(columns={"bullet_row_id": "example_id", "bullet": "text"})
        cluster_df["cluster_id"] = cluster_id
        concept_map, concept_df, _ = await synthesize(
            cluster_df, doc_col="text", doc_id_col="example_id", model=chat_model,
            cluster_id_col="cluster_id", n_concepts=SYNTH_N_CONCEPTS, pattern_phrase="unique topic",
            prompt_template=prompts_v2.SYNTHESIZE_PROMPT, return_concepts_dict=True,
        )
        if concept_map:
            concept_map, concept_df = await review(concept_map, concept_df, "concept", chat_model)
        rows = []
        for ordinal, c in enumerate(sorted(concept_map.values(), key=lambda c: (c.name, c.prompt))):
            rows.append({
                "concept_id": f"concept_{cp.content_hash({'run': run_id, 'cluster': cluster_id, 'name': c.name, 'prompt': c.prompt})[:24]}",
                "cluster_run_id": run_id, "cluster_id": cluster_id, "concept_ordinal": ordinal,
                "concept_label": c.name, "concept_criterion": c.prompt,
                "representative_bullet_ids": sorted(str(v) for v in c.example_ids),
                "synthesize_prompt_version": SYNTH_INFO["synthesize_prompt_version"],
                "synthesize_prompt_hash": SYNTH_INFO["synthesize_prompt_hash"],
            })
        return cluster_id, rows

tasks = [asyncio.create_task(synthesize_cluster(cluster_id, group)) for cluster_id, group in pending_groups]
progress = tqdm(total=len(tasks), desc="Synthesizing concepts", unit="cluster")
try:
    for task in asyncio.as_completed(tasks):
        cluster_id, rows = await task
        if rows:
            concept_rows.extend(rows)
            concepts = (pd.DataFrame(concept_rows).drop_duplicates("concept_id", keep="last")
                        .sort_values(["cluster_run_id", "cluster_id", "concept_ordinal", "concept_id"], kind="stable")
                        .reset_index(drop=True))
            cp.write_parquet(concepts, path("concepts"))
        progress.update(1)
        progress.set_postfix(concepts=len(concept_rows), cluster=cluster_id)
except BaseException:
    for task in tasks:
        task.cancel()
    await asyncio.gather(*tasks, return_exceptions=True)
    raise
finally:
    progress.close()

if concepts is None:
    concepts = pd.DataFrame(concept_rows)
if len(concepts):
    display(concepts[["cluster_id", "concept_label", "concept_criterion"]].head(20))
else:
    print("No concepts synthesized yet.")

chosen: E_umap15x5_mcs4_leaf_seed42_8425a39a5859 | clusters: 521


Synthesizing concepts:   0%|          | 0/521 [00:00<?, ?cluster/s]

,cluster_id,concept_label,concept_criterion
0,0,ביטול צוואה הדדית בחיים מחייב מתן הודעה בכתב ל...,האם הטקסט קובע כי ביטול צוואה הדדית בעוד בני ה...
1,1,איסור שביתה או השבתה בכיתה א' או בגן חובה במוס...,האם הטקסט קובע איסור על שביתה או השבתה בכיתה א...
2,2,איסור נטישת בעל חיים בידי בעליו או מחזיקו,האם הטקסט קובע איסור פלילי או משפטי על בעלים א...
3,3,סמכות המועצה לחייב מגדדים או קבלנים מורשים בהפ...,האם הטקסט קובע את סמכותה של המועצה לחייב מגדדי...
4,4,חובת צירוף תעודת רופא או תצהיר מיילדת בהודעה ע...,האם הטקסט קובע חובה לצרף תעודת רופא או תצהיר מ...
5,5,החלת תנאי שכר ותשלומים של בעלי תפקידים בבתי די...,"האם הטקסט קובע החלה של שכר, תשלומים או הוראות ..."
6,6,"איסור החזקה, הובלה או המיתה של בעל חיים אלא בה...","האם הטקסט אוסר על החזקה, הובלה או המיתה של בעל..."
7,6,איסור פגיעה או גרימת סבל לבעל חיים במופע (הפיכ...,האם הטקסט קובע איסור על הפיכת בעל חיים או הפעל...
8,7,איסור פגיעה פיזית או גרימת סבל לבעלי חיים,"האם הטקסט מטיל איסור פלילי או משפטי על גרירה, ..."
9,8,דרישות לתחולת הוראות מניעת ריבוי נישואין על בנ...,האם הטקסט קובע דרישות מצטברות הנוגעות למגורים ...


## 9. Build query-ready bill-level marts

`cp.build_analysis` is the long pure-pandas aggregation. `bridge_bill_clusters` is
direct HDBSCAN membership per bill; `bridge_bill_concepts` inherits labels from each
bill's clusters; `bill_concepts` is one row per bill. Probability columns are
membership strengths, not scores.

In [27]:
if chosen is None:
    raise RuntimeError("No CHOSEN_CANDIDATE: exploration is active, so marts are not built.")
marts = cp.build_analysis(bills, chunks, bullets, chosen, concepts)
for name, frame in marts.items():
    cp.write_parquet(frame, path(name))
    print(f"{name}: {len(frame)} rows")

report = cp.quality_report(bills, bullets, chosen, marts)
cp.write_parquet(report, path("quality_report"))
display(report)

dim_clusters: 522 rows
dim_concepts: 540 rows
bridge_bill_clusters: 1773 rows
bridge_bill_concepts: 1429 rows
bill_concepts: 1000 rows


,check,violations,status
0,duplicate source bill_id,0,pass
1,bill_concepts not one row per bill,0,pass
2,duplicate bullet assignment,0,pass
3,bullets missing assignment,1,fail
4,concept from noise cluster,0,pass
5,concept references missing concept,0,pass
6,scoring column present,0,pass


## 10. Trace one concept back to source

In [28]:
bbc = marts["bridge_bill_concepts"]
if len(bbc):
    row = bbc.iloc[0]
    chunk = chunks[chunks["chunk_id"] == row["primary_evidence_chunk_id"]].iloc[0]
    print("CONCEPT: ", row["concept_label"])
    print("CRITERION:", row["concept_criterion"])
    print("RELATION:", row["relation_type"], "| cluster", row["cluster_id"])
    print("BILL:    ", row["bill_id"], "-", chunk["bill_title"])
    print("SOURCE:  ", chunk["document_format"], "| hash", chunk["content_sha256"][:12])
    print("SOURCE (excerpt):", (chunk["text"] or "")[:400])
else:
    print("No concepts yet.")

CONCEPT:  הסמכת חברות ממשלתיות וישראליות מוגנות לפעול כחברות פיתוח, נכסים ונמל
CRITERION: האם הטקסט עוסק בהסמכת חברה בבעלות ממשלתית מלאה או חברה ישראלית מוגנות לשמש כחברת פיתוח ונכסים או כחברת נמל?
RELATION: representative | cluster 486
BILL:     15752 - חוק רשות הספנות והנמלים, התשס"ד-2004
SOURCE:   pdf | hash 89ed13ef6a2f
SOURCE (excerpt): "דיני הספנות והנמלים" ­ הפקודות והחוקים במפורט להלן, וכן התקנות שהותקנו לפיהם:

(ו) חוק זה;

פקודת הנמלים ננוסח חדש] ,התשל''א 1 1971­(להלן ­ פקודת הנמלים);

(ג) חוק הספנות (ימאים), התשל"ג21973­ (להלן ­ חוק הספנות (ימאים));

חוק הספנות (כלי שיט) ,התש"ך 3 1960­(להלן ­ חוק הספנות (כלי שיט)):

כל חוק אחר המסדיר תחום מתחומי הספנות והנמלים, שהשר ממונה על ביצועו,­

"הממונה על הנמלים" ,"מנהל נמל" ו''נמל" 
